# Event Exploration (Application)

Parsing the data and understanding it (EventData attribute)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from anomaly_detection.etl.load import load_records, load_eventdata_df, load_eventdata
from anomaly_detection.utils.profile_eventdata import profile_all, print_summary, summarize_constants, export_profiles_csv

plt.style.use('ggplot')

In [ ]:
# Path to the data
project_folder = Path.cwd().parent

output_path = project_folder / "data/event_exploration/application"

evtx_path = project_folder / "data/raw/93_applog.evtx"

evtx_path

In [ ]:
records = load_records(evtx_path)

len(records)

In [ ]:
with open(project_folder / "data/processed/record.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        records[0],
        indent=4
    ))

records[0]

In [ ]:
event_df = load_eventdata_df(evtx_path)

event_df.head()

In [ ]:
event_df.columns

In [ ]:
counts = event_df["EventID"].value_counts()

min_frequency = 254

frequent = counts[counts >= min_frequency]
others = counts[counts < min_frequency]

In [ ]:
event_descriptions = {
    4: "Generic PHP log message",
}

frequent.index = [
    f"{eid}: {event_descriptions.get(eid, 'Unknown')}"
    if isinstance(eid, int) else eid
    for eid in frequent.index
]

other_label = f""
frequent[other_label] = others.sum()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

wedges, label_texts = ax.pie(
    frequent.values,
    labels=frequent.index,
    startangle=90,
    labeldistance=0.45,
)

# Make EventIDs bold
for t in label_texts:
    t.set_fontweight("bold")
    t.set_fontsize(10)

ax.set_title("Event ID Distribution")
ax.axis("equal")
plt.tight_layout()

plt.savefig(output_path / "event_id_distribution.png")

plt.show()

In [ ]:
event_dfs = load_eventdata(evtx_path)

In [ ]:
profiles = profile_all(event_dfs)

export_profiles_csv(profiles, output_path)

In [ ]:
from contextlib import redirect_stdout

with open(output_path / 'profile_summary.txt', 'w') as f:
    with redirect_stdout(f):
        print_summary(profiles, event_dfs)

print_summary(profiles, event_dfs)

In [ ]:
constants = summarize_constants(profiles, event_dfs)

constants